<a href="https://colab.research.google.com/github/ailab-nda/ML/blob/main/Z_Image.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Image Generation with Diffusion Transformer

## Installing Relevant Software

In [ ]:
!git clone https://github.com/comfyanonymous/ComfyUI

%cd /content/ComfyUI
!pip install -r requirements.txt

!apt -y install -qq aria2

!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/T5B/Z-Image-Turbo-FP8/resolve/main/z-image-turbo-fp8-e4m3fn.safetensors -d /content/ComfyUI/models/diffusion_models -o z-image-turbo-fp8-e4m3fn.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors -d /content/ComfyUI/models/clip -o qwen_3_4b.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors -d /content/ComfyUI/models/vae -o ae.safetensors

## Defining an image generation function

In [ ]:
%cd /content/ComfyUI

import os, random, time

import torch
import numpy as np
from PIL import Image

from nodes import NODE_CLASS_MAPPINGS

UNETLoader = NODE_CLASS_MAPPINGS["UNETLoader"]()
CLIPLoader = NODE_CLASS_MAPPINGS["CLIPLoader"]()
VAELoader = NODE_CLASS_MAPPINGS["VAELoader"]()
CLIPTextEncode = NODE_CLASS_MAPPINGS["CLIPTextEncode"]()
KSampler = NODE_CLASS_MAPPINGS["KSampler"]()
VAEDecode = NODE_CLASS_MAPPINGS["VAEDecode"]()
EmptyLatentImage = NODE_CLASS_MAPPINGS["EmptyLatentImage"]()

with torch.inference_mode():
    unet = UNETLoader.load_unet("z-image-turbo-fp8-e4m3fn.safetensors", "fp8_e4m3fn_fast")[0]
    clip = CLIPLoader.load_clip("qwen_3_4b.safetensors", type="lumina2")[0]
    vae = VAELoader.load_vae("ae.safetensors")[0]

@torch.inference_mode()
def generate(input):
    tmp_dir="/content/ComfyUI/output"
    os.makedirs(tmp_dir, exist_ok=True)

    values = input["input"]

    positive_prompt = values['positive_prompt']
    negative_prompt = values['negative_prompt']
    seed = values['seed'] # 0
    steps = values['steps'] # 9
    cfg = values['cfg'] # 1.0
    sampler_name = values['sampler_name'] # euler
    scheduler = values['scheduler'] # simple
    denoise = values['denoise'] # 1.0
    width = values['width'] # 1024
    height = values['height'] # 1024
    batch_size = values['batch_size'] # 1.0

    if seed == 0:
        random.seed(int(time.time()))
        seed = random.randint(0, 18446744073709551615)

    positive = CLIPTextEncode.encode(clip, positive_prompt)[0]
    negative = CLIPTextEncode.encode(clip, negative_prompt)[0]
    latent_image = EmptyLatentImage.generate(width, height, batch_size=batch_size)[0]
    samples = KSampler.sample(unet, seed, steps, cfg, sampler_name, scheduler, positive, negative, latent_image, denoise=denoise)[0]
    decoded = VAEDecode.decode(vae, samples)[0].detach()
    Image.fromarray(np.array(decoded*255, dtype=np.uint8)[0]).save(f"{tmp_dir}/z_image_turbo.png")

    result = f"{tmp_dir}/z_image_turbo.png"

    return result

## Image Generation

In [ ]:
input = {
  "input": {
    "positive_prompt": "Naming a specific studio style (“KyoAni”) sets a precise aesthetic without copying a single title. The atmospheric cues – golden hour backlight, wind in hair, gradient sky – are iyashikei hallmarks that Z-Image interprets with strong fidelity. This prompt also demonstrates that Z-Image handles stylized illustration as well as photorealism.",
    #"positive_prompt": "Candid full-body photograph of a stylish young man walking down a clean Tokyo backstreet at dusk, wearing an oversized black bomber jacket and tailored trousers, neon glow reflecting softly on his face, shot on a 50mm lens, urban editorial vibe, sharp focus.",
    #"positive_prompt": {  "aspect_ratio": "2:3 (portrait)",  "scene_description": "An intimate, internet-aesthetic indoor portrait set in a minimalist living-room corner, where a single seated subject dominates the frame with a soft, flirtatious wink and relaxed posture; the image prioritizes cozy domestic texture—blue upholstered seating, pale walls, and a modern floor lamp—rendered with shallow depth-of-field and a bright, airy, slightly dreamy highlight bloom that reads like a polished social-media lifestyle shot.",  "prompt": {    "scene_and_composition": {      "visual_hook": "The immediate hook is the close, chest-up framing combined with the subject’s one-eye wink and slightly parted glossy lips, amplified by the off-shoulder white shirt sliding down to reveal thin straps and the centered drawstring detail; loose flyaway hair strands cut across the face, adding candid ‘caught-in-a-moment’ internet intimacy.",      "camera_angle": "Eye-level to slightly above-eye-level angle, positioned very close to the subject (arm’s-length selfie distance or short-lens portrait distance), with the camera pointed slightly downward so the face and neckline become the compositional apex; the subject is oriented frontally toward the lens with a small head tilt that creates a gentle S-curve through neck and shoulders.",      "framing_and_depth": "Tight portrait crop (head and upper torso dominate ~70% of the frame). Strong subject-background separation via shallow depth of field: the face and front hair strands are sharpest, while the blue sofa and floor lamp behind fall into a smooth, creamy blur; foreground is minimal, implying the subject is seated close to the camera with little intervening space.",      "medium_physics": "High-resolution digital portrait look with soft skin smoothing and bright highlight roll-off; mild lens bloom/veiling flare in the upper-left corner, subtle compression artifacts typical of social platforms, very light noise in the shadows, and no noticeable motion blur—only a few crisp flyaway hairs implying still air or minimal movement."    },    "subject_content": {      "overall_pose": "Seated, slightly leaning forward with shoulders relaxed and collarbones open, head subtly tilted; body squared toward the camera in a casual, intimate pose.",      "limb_articulation": "Arms are drawn inward across the front of the torso, elbows bent, creating a gentle self-hug/closed-in frame; hands are out of frame or tucked low. Legs are largely out of frame, implied seated on the floor or low surface near the sofa with knees bent.",      "pose_dynamics": "Weight is settled and relaxed; the forward lean adds proximity and tension-of-attention to the face, while the shoulders remain dropped and soft, producing a calm yet teasing, camera-aware energy.",      "makeup_look": "Soft-glam ‘clean’ makeup with a satin base, warm pink blush, lightly defined eyes with a soft liner/mascara effect, and a glossy, gradient-tinted lip; skin reads smooth with controlled highlights on cheekbones and nose bridge. (Makeup style is for reference only; must be merged with the input_photo’s facial traits.)",      "emotional_vibe": "Playful, intimate, slightly dreamy—an internet-friendly, approachable flirtiness created by the wink, relaxed mouth, and gentle head tilt without aggressive intensity.",      "main_object_apparel": "A white off-shoulder button-up shirt worn loosely and slipped down the shoulders, fabric appearing lightweight cotton with soft wrinkles and a matte finish; underneath is a white strappy camisole/bustier-like top with thin shoulder straps and a centered drawstring tie at the neckline, creating gathered pleats and a subtle focal point at the chest. (If this apparel is crucial to the photo’s vibe, prioritize it; if not, adapt to input_photo. If input_photo lacks full-body context, strictly use this apparel to complete the scene.)",      "hair_and_styling": "Medium-length dark hair tied back loosely with face-framing strands and wispy bangs; several flyaway strands cross the forehead and cheeks for a candid, slightly messy-chic look; small stud earrings as minimal accessories. (Hair styling is for reference only and should adapt to input_photo.)",      "interaction_with_environment": "The subject sits close to the blue sofa, using the space as a soft, domestic backdrop; posture suggests they are on the floor or a low spot in front of the couch, leaning forward into the camera’s personal space while the lamp and wall remain distant, creating a cozy corner vignette."    },    "environment": {      "visual_depth_stack": {        "foreground": [          "1) The subject’s white shirt fabric and neckline area occupying the lower half of frame",          "2) A soft, bright flare/bloom region in the upper-left corner washing the edge of the scene"        ],        "middle_ground": [          "1) A blue upholstered sofa backrest spanning the width behind the subject",          "2) A blue seat cushion area to the right side, slightly darker due to shadow",          "3) A pale floor surface at the lower-left edge, softly lit and out of focus"        ],        "background": [          "1) A light-colored wall with gentle tonal gradation",          "2) A modern floor lamp on the upper-right with a pale lampshade and dark rim, partially cropped",          "3) A faint vertical dark area on the far-left edge suggesting a doorway or furniture boundary"        ]      },      "lighting_atmosphere": {        "key_light": "Soft, bright key light coming from the upper-left/front-left, likely window light or diffused studio light, creating luminous highlights on the left side of the face and shoulder with smooth, non-harsh shadow transitions.",        "fill_ambient": "Even ambient fill from the room bounces light into the shadow side, keeping contrast low; shadows lean slightly cool-neutral while highlights remain warm-neutral, producing a gentle, airy separation without dramatic chiaroscuro.",        "interaction": "Highlight bloom/veiling flare in the top-left lifts the scene’s perceived softness; subtle rim-like glow appears along hair strands near the lit side; fabric wrinkles catch small specular accents, while the sofa absorbs light into a velvety, matte blue blur."      }    },    "realism_rules": {      "style_definition": [        "Modern Chinese social-media portrait aesthetic (polished yet ‘candid’)",        "Soft natural-light look with shallow depth of field and creamy bokeh",        "Clean, airy indoor lifestyle vibe with gentle highlight bloom"      ],      "imperfection_tolerance": [        "Allow mild lens flare/veiling haze in highlights",        "Allow slight skin retouching but keep natural texture hints (pores subtly implied, not plastic)",        "Allow minor compression artifacts and faint shadow noise consistent with platform uploads",        "Allow stray hair strands crossing the face; do not over-clean them"      ]    }  },  "single_paragraph_narrative": "A vertical 2:3 intimate indoor portrait centers a single seated subject in a cozy living-room corner, captured at eye-level to slightly above with a close, chest-up crop that makes the wink and glossy, softly parted lips the first-read internet hook, while loose flyaway bangs and face-framing strands cut across the face to preserve a candid, personal-space feel; the subject leans slightly forward with relaxed shoulders and a gentle head tilt, arms drawn inward in a self-framing posture, wearing a loosely draped white off-shoulder button-up with matte cotton wrinkles and a white thin-strap inner top featuring a gathered drawstring tie at the neckline, which becomes a secondary focal point under the shallow depth-of-field; the key light is soft and bright from the upper-left/front-left—window-like and diffused—creating warm-neutral highlights on the lit cheek and shoulder, with low-contrast cool-neutral fill keeping shadows open, and a subtle veiling flare in the upper-left that adds dreamy bloom; the color palette balances creamy skin tones and crisp whites against a velvety, defocused blue sofa spanning the midground, while the background resolves into a pale wall gradient and a partially cropped modern floor lamp on the upper-right, all rendered in high-resolution digital clarity with gentle retouching, smooth bokeh, slight platform-style compression, and minimal noise, producing an airy, soft-glam lifestyle mood with clear foreground–midground–background separation."},
     "negative_prompt": "blurry ugly bad",
    "width": 1024,
    "height": 1024,
    "batch_size": 1,
    "seed": 0,
    "steps": 9,
    "cfg": 1,
    "sampler_name": "euler",
    "scheduler": "simple",
    "denoise": 1.0,
  }
}

output = generate(input)
Image.open(output)

### Prompt Guide
https://ltx.io/blog/z-image-prompt-guide